# Day 2 Football detection, tracking and teams

Start a fresh Colab runtime and select **T4 GPU** under **Runtime > Change runtime type**. In Colab's Secrets panel, add `GITHUB_TOKEN` with read access to the private `YuvanJain/sports-vision-tracking` repository and grant this notebook access. Setup clones `main` into local runtime storage, links the football video from Drive, and saves both trials to Drive. Run the 15-second baseline first, then the configurable 60-second trial.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

from google.colab import drive, userdata

ROOT = Path('/content/sports-vision-tracking')
DRIVE_ROOT = Path('/content/drive/MyDrive/Sports Vision Tracking')
FOOTBALL = DRIVE_ROOT / 'Data/Football.mp4'
OUTPUTS = DRIVE_ROOT / 'outputs/football'
if ROOT.exists():
    raise RuntimeError(f'{ROOT} already exists. Start a fresh T4 runtime and rerun setup.')
gpus = subprocess.check_output(['nvidia-smi', '-L'], text=True)
assert 'T4' in gpus, f'Select a T4 GPU runtime. Detected GPUs: {gpus.strip()}'
drive.mount('/content/drive')
assert FOOTBALL.is_file(), f'Football video missing at {FOOTBALL}'

# Pass the secret only to the clone subprocess. Keep the remote URL credential-free.
credential_helper = (
    '!f() { if [ "$1" = get ]; then printf "username=x-access-token\npassword=%s\n" "$GITHUB_TOKEN"; fi; }; f'
)
subprocess.run(
    ['git', '-c', 'credential.helper=', '-c', f'credential.helper={credential_helper}',
     'clone', '--branch', 'main', '--single-branch',
     'https://github.com/YuvanJain/sports-vision-tracking.git', str(ROOT)],
    env={**os.environ, 'GITHUB_TOKEN': userdata.get('GITHUB_TOKEN'), 'GIT_TERMINAL_PROMPT': '0'},
    check=True,
)

video_link = ROOT / 'Data/Football.mp4'
video_link.parent.mkdir(parents=True, exist_ok=True)
video_link.symlink_to(FOOTBALL)
assert video_link.is_symlink() and video_link.resolve(strict=True) == FOOTBALL.resolve(strict=True)
requirements = ROOT / 'requirements.txt'
assert requirements.is_file() and (ROOT / 'scripts/track_football.py').is_file()
OUTPUTS.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print(f'Repository: {ROOT}\nFootball video: {video_link} -> {FOOTBALL}\nOutputs: {OUTPUTS}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Switch this notebook to a GPU runtime and rerun setup'
print(torch.cuda.get_device_name(0))

def run_trial(name, seconds, start_frame, ball_fallback=False):
    output = OUTPUTS / name
    command = [
        sys.executable, 'scripts/track_football.py',
        '--device', 'cuda:0', '--start-frame', str(start_frame),
        '--seconds', str(seconds), '--output-dir', str(output),
    ]
    if ball_fallback:
        command.append('--ball-fallback')
    subprocess.run(command, check=True)
    metrics = json.loads((output / 'metrics.json').read_text())
    assert metrics['frames_processed'] == round(seconds * 25)
    for artifact in ('annotated.mp4', 'tracks.json', 'tracks.csv'):
        assert (output / artifact).stat().st_size > 0
    print(f'Artifacts saved to {output}')
    return metrics

## 15-second smoke trial

Open the resulting MP4 to inspect IDs, labels, and bounding boxes before running the longer trial.

In [ ]:
smoke_metrics = run_trial('smoke', seconds=15, start_frame=1500)

## Configurable one-minute trial

Use the same start and duration with the fallback disabled if you want to compare throughput and ball detection coverage. Inspect ball detections visually for false positives.

In [ ]:
START_FRAME = 1500
SECONDS = 60
BALL_FALLBACK = True

trial_metrics = run_trial('one_minute', SECONDS, START_FRAME, BALL_FALLBACK)